In [ ]:
# ==========================================
# 1. HAZIRLIK VE GÜVENLİ VERİ ÇIKARMA
# ==========================================
from google.colab import drive
import os
import shutil
import zipfile

# Drive'ı bağla
drive.mount('/content/drive')

# Kütüphaneleri kur
!pip install -q timm albumentations tqdm

# LÜTFEN BURAYI KENDİ ZİP DOSYANA GÖRE DÜZENLE!
ZIP_YOLU = "/content/drive/MyDrive/plantvillage dataset.zip"
DATA_DIR = "/content/plant_dataset/plantvillage dataset"

if not os.path.exists(DATA_DIR):
    print("Veri seti kopyalanıyor ve çıkarılıyor...")
    if os.path.exists(ZIP_YOLU):
        shutil.copy(ZIP_YOLU, "/content/dataset.zip")
        with zipfile.ZipFile("/content/dataset.zip", 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("Veri seti başarıyla çıkarıldı!")
    else:
        raise FileNotFoundError(f"HATA: {ZIP_YOLU} bulunamadı! Lütfen yolu kontrol et.")
else:
    print("Veri seti zaten çıkarılmış, eğitime geçiliyor.")

# ==========================================
# 2. KÜTÜPHANELER VE A100 OPTİMİZASYONLARI
# ==========================================
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import datasets
from torch.utils.data import DataLoader, random_split, Dataset
import timm
import numpy as np
from tqdm import tqdm

# A100 TF32 Hızlandırıcıları
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan Cihaz: {DEVICE}")

# ==========================================
# 3. HİPERPARAMETRELER VE VERİ ARTIRIMI
# ==========================================
BATCH_SIZE = 128  # B3 hafif olduğu için A100'de 128 veya 256 çok rahat çalışır
IMG_SIZE = 300    # EfficientNet-B3 için ideal ve orijinal çözünürlük
EPOCHS = 15

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.RandomShadow(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class PlantDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(image=np.array(x))["image"]
        return x, y

    def __len__(self):
        return len(self.subset)

# Veriyi yükle ve böl
full_dataset = datasets.ImageFolder(DATA_DIR)
NUM_CLASSES = len(full_dataset.classes)
print(f"Toplam Sınıf Sayısı: {NUM_CLASSES}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(PlantDataset(train_data, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(PlantDataset(val_data, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

# ==========================================
# 4. MODEL (EFFICIENTNET-B3) VE EĞİTİM ARAÇLARI
# ==========================================
print("EfficientNet-B3 modeli yükleniyor...")
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)

# Focal Loss (Sınıf dengesizliği için)
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5) # B3 için Learning Rate biraz daha yüksek tutulabilir
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ==========================================
# 5. EĞİTİM DÖNGÜSÜ
# ==========================================
best_val_acc = 0.0
print("🚀 Eğitim Başlıyor...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, leave=True)
    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        # A100 için donanımsal Bfloat16
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(loss=loss.item(), acc=100.*correct/total)

    scheduler.step()

    # DOĞRULAMA (VALIDATION)
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total
    print(f"\nDeğerlendirme -> Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%\n")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_path = "/content/drive/MyDrive/best_plant_model_efficientnet_b3.pth"
        torch.save(model.state_dict(), save_path)
        print(f"🏆 Yeni en iyi model Drive'a kaydedildi! -> {save_path}")

print("Eğitim Süreci Tamamlandı!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Veri seti zaten çıkarılmış, eğitime geçiliyor.
Kullanılan Cihaz: cuda
Toplam Sınıf Sayısı: 33
EfficientNet-B3 modeli yükleniyor...
🚀 Eğitim Başlıyor...


  0%|          | 0/250 [00:00<?, ?it/s]/tmp/ipython-input-3092/189016597.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
Epoch [1/15]: 100%|██████████| 250/250 [01:36<00:00,  2.59it/s, acc=91.2, loss=0.0208]
/tmp/ipython-input-3092/189016597.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):



Değerlendirme -> Val Loss: 0.0259, Val Acc: 98.26%

🏆 Yeni en iyi model Drive'a kaydedildi! -> /content/drive/MyDrive/best_plant_model_efficientnet_b3.pth


Epoch [2/15]: 100%|██████████| 250/250 [01:36<00:00,  2.59it/s, acc=97.2, loss=0.0163]



Değerlendirme -> Val Loss: 0.0374, Val Acc: 97.09%



...


Değerlendirme -> Val Loss: 0.0049, Val Acc: 99.73%

🏆 Yeni en iyi model Drive'a kaydedildi! -> /content/drive/MyDrive/best_plant_model_efficientnet_b3.pth


Epoch [14/15]: 100%|██████████| 250/250 [01:36<00:00,  2.59it/s, acc=99.9, loss=4.6e-6]



Değerlendirme -> Val Loss: 0.0046, Val Acc: 99.73%



Epoch [15/15]: 100%|██████████| 250/250 [01:36<00:00,  2.60it/s, acc=100, loss=0.000249]



Değerlendirme -> Val Loss: 0.0048, Val Acc: 99.70%

Eğitim Süreci Tamamlandı!


In [ ]:
# Eğitim bittikten sonra sınıfların gerçek sırasını kontrol et

# Hata düzeltme: 'full_dataset' değişkeni tanımlı olmadığı için hata alınıyor.
# Kernel yeniden başlatıldığında önceki hücrelerdeki tanımlamalar kaybolur.
# Bu hücreyi bağımsız hale getirmek için gerekli tanımlamaları buraya ekliyoruz.

import os
from torchvision import datasets

# DATA_DIR değişkeni, önceki hücrelerde tanımlanmıştı. Buraya tekrar ekliyoruz.
DATA_DIR = "/content/plant_dataset/plantvillage dataset"

# full_dataset nesnesini yeniden oluşturuyoruz.
# Eğer DATA_DIR mevcut değilse, bir uyarı verilir.
if os.path.exists(DATA_DIR):
    full_dataset = datasets.ImageFolder(DATA_DIR)
    print("Modelin Eğitildiği Gerçek Sıralama:")
    for idx, cls in enumerate(full_dataset.classes):
        print(f"{idx}: {cls}")
else:
    print(f"HATA: Veri dizini bulunamadı: {DATA_DIR}. Lütfen ZIP dosyasının çıkarıldığından ve yolun doğru olduğundan emin olun.")

# Bu çıktıyı kopyala ve tahmin (inference) kodundaki class_names listesiyle karşılaştır.


HATA: Veri dizini bulunamadı: /content/plant_dataset/plantvillage dataset. Lütfen ZIP dosyasının çıkarıldığından ve yolun doğru olduğundan emin olun.


In [ ]:
import os
import shutil
import zipfile

# Önceki hücrelerde tanımlanan yolları tekrar kullanıyoruz
# LÜTFEN BURAYI KENDİ ZİP DOSYANA GÖRE DÜZENLE! (Gerekirse)
ZIP_YOLU = "/content/drive/MyDrive/plantvillage dataset.zip"
DATA_DIR = "/content/plant_dataset/plantvillage dataset"

print(f"Zip dosya yolu: {ZIP_YOLU}")
print(f"Verilerin çıkarılacağı dizin: {DATA_DIR}")

# Eğer DATA_DIR zaten varsa, yeniden çıkarmadan önce temizle
if os.path.exists(DATA_DIR):
    print(f"{DATA_DIR} zaten mevcut, siliniyor ve yeniden oluşturuluyor...")
    shutil.rmtree(DATA_DIR)
    os.makedirs(DATA_DIR)
else:
    os.makedirs(DATA_DIR, exist_ok=True)


if os.path.exists(ZIP_YOLU):
    print("Veri seti kopyalanıyor ve çıkarılıyor...")
    # Colab'da Drive'dan büyük dosyaları doğrudan çıkarmak yerine,
    # önce /content'a kopyalamak genellikle daha güvenlidir ve hızlıdır.
    local_zip_path = "/content/dataset.zip"
    shutil.copy(ZIP_YOLU, local_zip_path)

    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Veri seti başarıyla çıkarıldı!")

    # Kopyalanan zip dosyasını sil
    os.remove(local_zip_path)
else:
    raise FileNotFoundError(f"HATA: {ZIP_YOLU} bulunamadı! Lütfen yolu kontrol edin ve Drive'ınızda olduğundan emin olun.")


Zip dosya yolu: /content/drive/MyDrive/plantvillage dataset.zip
Verilerin çıkarılacağı dizin: /content/plant_dataset/plantvillage dataset


FileNotFoundError: HATA: /content/drive/MyDrive/plantvillage dataset.zip bulunamadı! Lütfen yolu kontrol edin ve Drive'ınızda olduğundan emin olun.

In [ ]:
import os
import shutil

# 1. Mac'in oluşturduğu gereksiz klasörü kökten sil
macosx_dir = "/content/plant_dataset/__MACOSX"
if os.path.exists(macosx_dir):
    shutil.rmtree(macosx_dir)
    print("🧹 __MACOSX klasörü başarıyla silindi.")

# 2. Asıl verilerin olduğu doğru klasör yolunu belirle
DATA_DIR = "/content/plant_dataset/plantvillage dataset"
print(f"📁 Yeni ve doğru DATA_DIR yolu: {DATA_DIR}")

# 3. Klasörlerin içine sızmış gizli (._) önbellek dosyalarını temizle
!find "/content/plant_dataset" -name "._*" -type f -delete
print("✨ Tüm gizli macOS önbellek dosyaları temizlendi!")

🧹 __MACOSX klasörü başarıyla silindi.
📁 Yeni ve doğru DATA_DIR yolu: /content/plant_dataset/plantvillage dataset
✨ Tüm gizli macOS önbellek dosyaları temizlendi!
